# 02 · Image preprocessing and PyTorch data pipeline

This notebook prepares the cleaned blood-cell images for the modelling stages.

The train, validation and test splits were created in **Notebook 1** and saved as CSV files. Here, I load those existing splits, check that the image paths are valid, standardise the image dimensions and build a PyTorch pipeline that can serve images to a model in batches.

The aim is to keep the original cell morphology as intact as possible while making the data consistent and easy to work with.

## 1. Load the existing data splits

I reuse the CSV files generated in the previous notebook rather than cleaning or splitting the data again. This keeps the data preparation process reproducible and ensures that later models use the same train, validation and test sets.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

PROCESSED_DIR = Path("../data/processed")
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print(f"Training images:   {len(train_df):,}")
print(f"Validation images: {len(val_df):,}")
print(f"Testing images:    {len(test_df):,}")
print(f"Total images:      {len(train_df) + len(val_df) + len(test_df):,}")

## 2. Check the splits and image paths

Before preprocessing, I check the class counts in each split and confirm that every path points to an existing image file. The class distributions should remain similar because the split in Notebook 1 was stratified.

In [ ]:
for name, dataframe in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df),
]:
    print(f"\\n{name} class distribution:")
    print(dataframe["class"].value_counts().sort_index())

In [ ]:
for name, dataframe in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df),
]:
    missing = sum(not Path(path).is_file() for path in dataframe["path"])
    print(f"{name} — missing image files: {missing}")

## 3. Inspect an example image

The source images are close to square, but their dimensions are not all identical. I inspect one training image first to confirm its dimensions and colour mode before deciding how to standardise it.

In [ ]:
sample_path = train_df.iloc[0]["path"]

with Image.open(sample_path) as source_image:
    image = source_image.convert("RGB")

print("Original dimensions:", image.size)
print("Image mode:", image.mode)

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.title("Original blood cell image")
plt.axis("off")
plt.show()

## 4. Resize images while preserving aspect ratio

The models will expect images with consistent dimensions. I use `thumbnail()` to scale each image so that it fits inside a 128 × 128 pixel frame without stretching it. I then place it in the centre of a square canvas.

This approach preserves the image's aspect ratio. Any unused space is padded with black pixels. The same operation will be applied to every image, so the model receives a consistent input shape.

In [ ]:
def resize_and_pad(image_path, target_size=IMAGE_SIZE):
    """Resize an image proportionally and centre it on a fixed-size canvas."""
    with Image.open(image_path) as source_image:
        image = source_image.convert("RGB")

    image.thumbnail(target_size, Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", target_size, (0, 0, 0))
    x_offset = (target_size[0] - image.width) // 2
    y_offset = (target_size[1] - image.height) // 2
    canvas.paste(image, (x_offset, y_offset))

    return canvas

In [ ]:
resized_image = resize_and_pad(sample_path)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(image)
axes[0].set_title(f"Original · {image.size[0]} × {image.size[1]}")
axes[0].axis("off")

axes[1].imshow(resized_image)
axes[1].set_title(f"Resized · {resized_image.size[0]} × {resized_image.size[1]}")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Convert pixel values to the range [0, 1]

An image opened with PIL contains integer pixel values between 0 and 255. I convert the image to a NumPy array with `float32` values and divide by 255, scaling the pixel range to [0, 1].

This is a common input representation for neural networks and avoids passing large integer pixel values directly to the model.

In [ ]:
image_array = np.asarray(resized_image)

normalized_image = image_array.astype(np.float32) / 255.0

print("Array shape (height, width, channels):", normalized_image.shape)
print("Data type:", normalized_image.dtype)
print("Pixel range:", normalized_image.min(), "to", normalized_image.max())

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(normalized_image)
plt.title("Resized and normalized image")
plt.axis("off")
plt.show()

## 6. Build a PyTorch Dataset

Rather than loading every preprocessed image into memory at once, I use a custom `Dataset`. It reads an image from its file path and applies the same resizing and normalization steps when that sample is requested.

The dataset also converts each class name into an integer label. PyTorch classification models use integer class indices when training with common loss functions such as cross-entropy.

The image is converted from **H × W × C** (the usual NumPy/PIL layout) to **C × H × W**, which is the layout expected by PyTorch image models.

In [ ]:
classes = sorted(train_df["class"].unique())

class_to_idx = {class_name: idx for idx, class_name in enumerate(classes)}
idx_to_class = {idx: class_name for class_name, idx in class_to_idx.items()}

print("Class-to-index mapping:")
print(class_to_idx)

In [ ]:
class BloodCellDataset(Dataset):
    """Load and preprocess blood-cell images and their class labels."""

    def __init__(self, dataframe, class_to_idx, image_size=IMAGE_SIZE):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.image_size = image_size

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["path"]) as source_image:
            image = source_image.convert("RGB")

        image.thumbnail(self.image_size, Image.Resampling.LANCZOS)

        canvas = Image.new("RGB", self.image_size, (0, 0, 0))
        x_offset = (self.image_size[0] - image.width) // 2
        y_offset = (self.image_size[1] - image.height) // 2
        canvas.paste(image, (x_offset, y_offset))

        image_array = np.asarray(canvas, dtype=np.float32) / 255.0
        image_tensor = torch.from_numpy(image_array).permute(2, 0, 1)

        label = self.class_to_idx[row["class"]]
        label_tensor = torch.tensor(label, dtype=torch.long)

        return image_tensor, label_tensor

## 7. Create the datasets and data loaders

I create one dataset for each split. The `DataLoader` wraps each dataset and returns batches of 32 images and labels.

Training data is shuffled so that the model does not see examples in the same order every epoch. Validation and test data are not shuffled; keeping their order fixed makes evaluation easier to reproduce.

`num_workers=0` is used for this local setup, which is also a straightforward option when working in a notebook on macOS.

In [ ]:
train_dataset = BloodCellDataset(train_df, class_to_idx)
val_dataset = BloodCellDataset(val_df, class_to_idx)
test_dataset = BloodCellDataset(test_df, class_to_idx)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

## 8. Select the available compute device

PyTorch can use different compute devices depending on the machine. On this Mac, I check for Apple's Metal Performance Shaders (MPS) backend first, then CUDA, and finally fall back to the CPU.

The selected device will be used when models and batches are moved to the device in the training notebook.

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("Selected device:", device)

## 9. Verify a batch

Before moving on to modelling, I retrieve one batch and check its shape, data type and pixel range.

The expected image tensor shape is **(batch size, channels, height, width)**. With a batch size of 32 and image size of 128 × 128, that gives `(32, 3, 128, 128)`. Labels should be a one-dimensional tensor of integer class indices.

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)
print("Pixel range:", images.min().item(), "to", images.max().item())

## 10. Visualise a batch and check its labels

Finally, I display eight images from a training batch with their decoded class names. This is a simple visual check that the images and labels are being returned together as expected.

In [ ]:
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for index, ax in enumerate(axes.flat):
    image = images[index].permute(1, 2, 0).numpy()
    label = idx_to_class[labels[index].item()]

    ax.imshow(image)
    ax.set_title(label)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Summary

The preprocessing and data pipeline are now in place:

- The train, validation and test CSV files from Notebook 1 load correctly.
- Images are resized proportionally and padded to 128 × 128 pixels.
- Pixel values are converted to `float32` and scaled to [0, 1].
- Class names are mapped to integer labels.
- A custom PyTorch `Dataset` loads and preprocesses images on demand.
- `DataLoader` creates batches for training, validation and testing.
- A sample batch has been checked visually and programmatically.

**Why load images on demand?** The earlier preprocessing checks used a small in-memory example to verify the transformation. For the full dataset, loading every image into large NumPy arrays would use considerably more memory. The `Dataset`/`DataLoader` approach reads samples as needed, which is more practical for model training.

The next notebook will establish a classical machine-learning baseline using handcrafted image features before comparing it with deep-learning approaches.